![CARD: encoder-free audio captioning](https://raw.githubusercontent.com/KarthikKolluriKB/CARD/main/assets/logo/card-social-preview.png)

# CARD: encoder-free audio captioning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KarthikKolluriKB/CARD/blob/main/notebooks/CARD_demo.ipynb)

Run **CARD** live on your own audio. CARD is the model from *"CARD: Cross-component Audio
Representation Distillation for Encoder-Free Audio Captioning"* (IEEE SLT 2026).

CARD has no audio encoder at inference. A 13.2 M-parameter projector turns the log-Mel spectrogram
into audio tokens, which a Qwen3-4B language model with merged LoRA adapters reads to write the caption.

- Model: [KarthikKB1998/CARD-Qwen3-4B-AudioCaps](https://huggingface.co/KarthikKB1998/CARD-Qwen3-4B-AudioCaps)
- Demo page: [KarthikKB1998/CARD-Audio-Captioning](https://huggingface.co/spaces/KarthikKB1998/CARD-Audio-Captioning)
- Code: [KarthikKolluriKB/CARD](https://github.com/KarthikKolluriKB/CARD)

**Before you start:** in Colab choose *Runtime > Change runtime type > T4 GPU* (or any GPU), then run
the cells from top to bottom. The first model load downloads about 8.4 GB and takes a few minutes.

## 1. Install

Installs the `card` package from GitHub and the library versions the model was released with.

In [ ]:
!pip install -q "transformers==4.53.1" soundfile
!pip install -q --no-deps "git+https://github.com/KarthikKolluriKB/CARD"
print("installed")

## 2. Check the GPU

CARD was trained and evaluated in bfloat16. The cell uses bfloat16 when the GPU can run it and
falls back to float16 otherwise, which can change a caption slightly.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        a = torch.randn(512, 512, device=DEVICE, dtype=torch.bfloat16)
        (a @ a).sum().item()
        DTYPE = torch.bfloat16
    except Exception:
        DTYPE = torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.bfloat16
    print("No GPU found. In Colab choose Runtime > Change runtime type > T4 GPU and run this cell again.")
    print("On CPU the model still works, but one caption can take minutes.")
print("device:", DEVICE, "| dtype:", DTYPE)

## 3. Load the model

Downloads the merged model from the Hugging Face Hub. To try the Clotho fine-tune instead, set
`MODEL_ID = "KarthikKB1998/CARD-Qwen3-4B-Clotho"`.

If the model repository is still private, first run `from huggingface_hub import notebook_login; notebook_login()`
and sign in with your own access token.

In [ ]:
import time
from card import load_card

MODEL_ID = "KarthikKB1998/CARD-Qwen3-4B-AudioCaps"

t0 = time.time()
model = load_card(MODEL_ID, device=DEVICE, dtype=DTYPE)
print(model)
print(f"loaded in {time.time() - t0:.0f} s")

## 4. Try the demo clips

Captions the three clips from the demo page and shows the human reference captions next to them.
Words the model shares with a reference are highlighted. The last line says whether this runtime
reproduces the captions of the released model exactly.

In [ ]:
import html
import json
import re

from huggingface_hub import hf_hub_download
from IPython.display import Audio, HTML, display

STOPWORDS = {"a", "an", "the", "and", "or", "is", "are", "be", "being", "in", "on", "of", "with", "as",
             "to", "by", "while", "then", "some", "it", "its", "at", "from", "for", "into", "followed",
             "there", "this", "that", "several", "multiple", "times", "time"}


def _stem(word):
    w = word.lower()
    return w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w


def show_caption(caption, references=(), title="", seconds=None, released=None):
    ref_words = {_stem(w) for r in references for w in re.findall(r"[A-Za-z]+", r)}
    parts = []
    for part in re.split(r"([A-Za-z]+)", caption):
        if re.fullmatch(r"[A-Za-z]+", part) and part.lower() not in STOPWORDS and _stem(part) in ref_words:
            parts.append(f"<mark style='background:rgba(139,92,246,.25);color:inherit;border-radius:4px;padding:0 3px'>{html.escape(part)}</mark>")
        else:
            parts.append(html.escape(part))
    info = []
    if seconds is not None:
        info.append(f"{seconds:.1f} s")
    if released is not None:
        info.append("identical to the released caption" if caption == released
                    else "differs from the released caption: " + html.escape(released))
    refs = "".join(f"<li>{html.escape(r)}</li>" for r in references)
    display(HTML(
        "<div style='border:1px solid rgba(128,128,128,.35);border-left:4px solid #6366f1;border-radius:12px;"
        "padding:12px 16px;margin:6px 0 18px;font-family:Inter,system-ui,sans-serif'>"
        f"<div style='font-size:12px;opacity:.7;text-transform:uppercase;letter-spacing:.06em'>{html.escape(title)}</div>"
        f"<div style='font-size:20px;font-weight:700;margin:6px 0'>{''.join(parts)}</div>"
        f"<div style='font-size:12px;opacity:.7'>{' | '.join(info)}</div>"
        + (f"<div style='margin-top:10px;font-size:12px;opacity:.7'>Human reference captions</div><ol style='margin:4px 0 0'>{refs}</ol>" if refs else "")
        + "</div>"))


SPACE_ID = "KarthikKB1998/CARD-Audio-Captioning"
try:
    samples = json.load(open(hf_hub_download(SPACE_ID, "samples/samples.json", repo_type="space")))["samples"]
except Exception as e:
    samples = []
    print("Could not fetch the demo clips from the Space:", type(e).__name__, e)

same = 0
for s in samples:
    path = hf_hub_download(SPACE_ID, "samples/" + s["file"], repo_type="space")
    display(Audio(path))
    t0 = time.time()
    caption = model.caption(path)
    released = s.get("expected_caption")
    same += caption == released
    show_caption(caption, s.get("references", []), s["name"], time.time() - t0, released)
if samples:
    print(f"{same}/{len(samples)} captions identical to the released model's captions")

## 5. Caption your own audio

Upload one or more audio files (wav, flac, mp3, ...). Any sample rate works; stereo is averaged to
mono. CARD was trained on 10 s clips, so only the first 10 s of a longer file are used.

In [ ]:
import os

try:
    from google.colab import files
    paths = list(files.upload())
except ImportError:  # running outside Colab: list your files here
    paths = []

for p in paths:
    if not os.path.exists(p):
        print("not found:", p)
        continue
    display(Audio(p))
    t0 = time.time()
    show_caption(model.caption(p), title=os.path.basename(p), seconds=time.time() - t0)

## 6. Options

- `model.caption(path)` uses the paper's decoding: beam search with 4 beams and at most 40 new tokens.
- `model.caption(path, num_beams=1)` decodes greedily. It is faster, and captions can differ slightly.
- `model.caption(waveform, sr=16000)` also accepts a NumPy array or tensor with its sample rate.

In [ ]:
# Example: greedy decoding on the last uploaded or demo clip
last = paths[-1] if paths else (hf_hub_download(SPACE_ID, "samples/" + samples[0]["file"], repo_type="space") if samples else None)
if last:
    t0 = time.time()
    show_caption(model.caption(last, num_beams=1), title="greedy decoding", seconds=time.time() - t0)

## Citation

```bibtex
@inproceedings{kolluri2026card,
  title     = {{CARD}: Cross-component Audio Representation Distillation for Encoder-Free Audio Captioning},
  author    = {Kolluri, Ganesh Pavan Kartikeya Bharadwaj and Zhang, Yuchen and Kampouridis, Michael and Shekhar, Ravi},
  booktitle = {Proceedings of the IEEE Spoken Language Technology Workshop (SLT)},
  year      = {2026}
}
```